# Verify Imported Colab BioEmu Graph Results

Run this notebook with the normal local project kernel after downloading and importing no-Drive Colab result archives. It verifies canonical local paths and can rerun graph analysis without BioEmu or a GPU.

## 1. Expected Local Layout

Each imported archive writes its portable bundle under `data/bioemu_outputs/first_five_proteins/` and graph products under `outputs/dataset_bioemu/`. Use `scripts/colab_transfer.py import-result` rather than extracting ZIPs manually.

In [ ]:
from pathlib import Path
import json

DATASET_ROOT = Path("data/dataset/first_five_proteins")
BUNDLES_ROOT = Path("data/bioemu_outputs/first_five_proteins")
OUTPUTS_ROOT = Path("outputs/dataset_bioemu")
REQUESTED_SAMPLES = 200

EXPECTED_PAIRS = [
    ("P69441", "4AKE_A__1AKE_A"),
    ("P06746", "1BPX_A__1BPY_A"),
    ("P0AEX9", "1OMP_A__1ANF_A"),
    ("P00889", "1CTS_A__2CTS_A"),
    ("P35557", "1V4T_A__1V4S_A"),
]

## 2. Check Import Status

In [ ]:
status_rows = []
for protein_id, pair_id in EXPECTED_PAIRS:
    run_name = f"n_{REQUESTED_SAMPLES}"
    pair_dir = DATASET_ROOT / protein_id / "pairs" / pair_id
    bundle_dir = BUNDLES_ROOT / protein_id / pair_id / run_name
    output_dir = OUTPUTS_ROOT / protein_id / pair_id / run_name
    row = {
        "protein_id": protein_id,
        "pair_id": pair_id,
        "dataset": (pair_dir / "manifest.json").exists(),
        "bundle": (bundle_dir / "manifest.json").exists(),
        "graph": (output_dir / "experiment_summary.json").exists(),
    }
    if row["bundle"]:
        manifest = json.loads((bundle_dir / "manifest.json").read_text(encoding="utf-8"))
        row["retained_samples"] = len(manifest.get("samples", []))
    status_rows.append(row)
status_rows

## 3. Validate One Imported Pair And Optionally Rerun Graph Search

This defaults to the smallest production pair. It strictly validates sequence correspondence and common-residue mapping before graph construction.

In [ ]:
from guided_conformational_path.dataset_bioemu_adapter import (
    load_adapted_bioemu_nodes,
    load_dataset_pair_selection,
)

PROTEIN_ID = "P69441"
PAIR_ID = "4AKE_A__1AKE_A"
RUN_NAME = f"n_{REQUESTED_SAMPLES}"
PAIR_DIR = DATASET_ROOT / PROTEIN_ID / "pairs" / PAIR_ID
BUNDLE_DIR = BUNDLES_ROOT / PROTEIN_ID / PAIR_ID / RUN_NAME

selection = load_dataset_pair_selection(PAIR_DIR)
nodes = load_adapted_bioemu_nodes(selection, BUNDLE_DIR, max_samples=REQUESTED_SAMPLES)
print(
    f"Validated {len(nodes)} nodes: {len(selection.selected_sequence)} full residues -> "
    f"{len(selection.residue_positions)} common endpoint residues"
)

In [ ]:
RERUN_GRAPH = False
if RERUN_GRAPH:
    from guided_conformational_path.dataset_bioemu_experiment import (
        DatasetBioEmuConfig,
        run_dataset_bioemu_experiment,
    )
    from guided_conformational_path.large_synthetic import format_run_summary, write_run_outputs

    config = DatasetBioEmuConfig(
        dataset_pair_dir=PAIR_DIR,
        bioemu_bundle_dir=BUNDLE_DIR,
        max_bioemu_samples=REQUESTED_SAMPLES,
        k_neighbors=10,
        outputs_dir=OUTPUTS_ROOT / PROTEIN_ID / PAIR_ID / RUN_NAME,
    )
    run = run_dataset_bioemu_experiment(config)
    write_run_outputs(run, config)
    print(format_run_summary(run, config.outputs_dir))

The graph path remains a heuristic structural proposal, not evidence of physical or kinetic connectivity.